# 505 — Kong Konnect Bootstrap and Connectivity

**Deployment model: Konnect Serverless Gateway.**  
This project uses Konnect's fully managed gateway — Kong provisions and
operates the data plane for you.  No containers to run, no TLS certificates
to generate, no servers to manage.

**Goal:** install decK, create a Konnect Personal Access Token (PAT),
export the required environment variables, and validate that your local
machine can reach the Konnect control plane — all before any live traffic
is routed through Kong.

**No live AI or MCP traffic is routed through Kong in this notebook.**  
That happens in notebooks 506 (AI Gateway) and 507 (MCP Gateway).

---

## Prerequisites

Before you begin you need:

| Requirement | Notes |
|---|---|
| A Kong Konnect account | Free tier is fine. Sign up at `cloud.konghq.com`. |
| `curl` and a shell | macOS / Linux / WSL2 |
| Python 3.11 and this repo's dependencies installed | `pip install -r requirements.txt` from the repo root |
| Your `.env` file set up for Neo4j and Anthropic | See README — Kong vars are **not** required for this notebook |

---

## How live tests are gated

Cells that reach out to Konnect over the network are gated behind an
environment variable:

```bash
ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true
```

If the variable is absent or `false`, those cells print a skip notice and
continue cleanly.  You can complete the entire setup walkthrough by reading
the notebook; then come back and run the live cells once your PAT is ready.

In [2]:
import os
import subprocess
import sys

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from src.config import get_kong_settings

kong = get_kong_settings()
LIVE = kong.notebook_live_tests

print(f"Kong settings loaded (masked): {kong.masked()}")
print(f"ENABLE_LIVE_KONG_NOTEBOOK_TESTS = {LIVE}")
if not LIVE:
    print("\n[INFO] Live tests are DISABLED. Network cells will be skipped.")
    print("       Set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true in .env to enable them.")

Kong settings loaded (masked): {'konnect_region': 'au', 'konnect_addr': 'https://au.api.konghq.com', 'konnect_control_plane_name': 'entity-risk-ai', 'konnect_token': 'kpat_LuC…***', 'konnect_control_plane_id': 'a92e7da1-469b-4cda-851a-eb326907df7c', 'notebook_live_tests': True}
ENABLE_LIVE_KONG_NOTEBOOK_TESTS = True


---

## 1. What is Kong Konnect?

### 1.1 The short version

Kong Konnect is a **hosted API management platform**.  It separates the
problem of *configuring* Kong from the problem of *running* Kong.

There are three deployment models inside Konnect.  **This project uses
Serverless.**

```
┌────────────────────────────────────────────────────────┐
│  Konnect (cloud.konghq.com)           CONTROL PLANE    │
│  - stores routes, services, plugins, consumers         │
│  - provides Admin API / UI / decK sync                 │
└──────────────────────────┬─────────────────────────────┘
                           │  config (HTTPS)
         ┌─────────────────▼──────────────────────────┐
         │  Serverless Gateway   DATA PLANE            │
         │  Kong manages this — no containers needed   │
         │  Kong provides a public proxy URL for you   │
         └────────────────────────────────────────────┘
```

### 1.2 What Serverless means in practice

| Responsibility | Who handles it |
|---|---|
| Routes, services, plugins, consumers | **You** — via decK YAML or Konnect UI |
| Data plane nodes, TLS, scaling, uptime | **Kong** — fully managed |
| Proxy URL | **Kong provides it** — shown in the Gateway Manager dashboard |
| Docker containers / Helm charts / VM setup | **Not required** |
| Cluster certificates / hybrid bootstrap | **Not required** |

### 1.3 Why Serverless for this project

- **No infrastructure overhead** — the focus is on routes, plugins, and
  policy configuration, not on running Kong nodes.
- **Matches the Railway deployment model** — the app is already hosted
  as a managed service; the gateway should be too.
- **Faster iteration** — push config with decK, test immediately, no
  container restart required.
- **Phase 1 architecture fits** — Serverless provides a stable proxy URL
  that `KONG_PROXY_URL` can point to in notebooks 506 and 507.

### 1.4 Konnect regions

Konnect is multi-region.  Choose the region closest to your deployment:

| Region key | Konnect URL |
|---|---|
| `eu` | `https://eu.api.konghq.com` |
| `us` | `https://us.api.konghq.com` |
| `au` | `https://au.api.konghq.com` |
| `in` | `https://in.api.konghq.com` |

decK uses these when you pass `--konnect-region au` (or the equivalent env var).
The region must match the region your Konnect org was created in — you cannot
change it after org creation.

---

## 2. Install and verify decK

**decK** is the official Kong declarative configuration tool.  It lets you
describe your entire Kong config in YAML files and sync them to Konnect
with a single command — no clicking required.

### 2.1 Installation

**macOS (Homebrew)**
```bash
brew install kong/deck/deck
```

**Linux (script)**
```bash
curl -sL https://github.com/Kong/deck/releases/latest/download/deck_linux_amd64.tar.gz \
  | tar xz -C /usr/local/bin deck
deck version
```

**Windows (WSL2)**  
Use the Linux script above inside your WSL2 terminal.

**Docker (no local install)**
```bash
docker run --rm kong/deck:latest version
```

### 2.2 Verify

After installing, run `deck version`.  You should see output like:

```
decK v1.40.0
  commit: abc1234
  built with: go1.22.0
```

Any version ≥ 1.35 supports Konnect with `--konnect-*` flags.

In [2]:
# Verify decK is installed and accessible.
# This is a local check — no network access, no LIVE guard needed.

result = subprocess.run(["deck", "version"], capture_output=True, text=True)
if result.returncode == 0:
    print("[PASS] deck found:")
    print(result.stdout.strip())
else:
    print("[FAIL] 'deck' command not found or returned an error.")
    print("       Install decK first — see the cell above for instructions.")
    print(result.stderr.strip())

[PASS] deck found:
decK v1.55.2 (8bfab74)


---

## 3. Create a Konnect Personal Access Token (PAT)

A PAT is the credential that decK (and later, the Kong Admin API) uses to
authenticate with your Konnect organisation.

### 3.1 Via the Konnect UI

1. Open `https://cloud.konghq.com` and sign in.
2. Click your **avatar / initials** in the top-right corner.
3. Choose **My Account**.
4. Scroll to the **Personal Access Tokens** section.
5. Click **Generate Token**.
6. Give it a name, e.g. `entity-risk-ai-dev`, set an expiry, and click **Create**.
7. **Copy the token immediately** — Konnect only shows it once.

### 3.2 Minimum required permissions

The PAT owner needs at least:

| Scope | Why |
|---|---|
| Control Planes → View | decK reads existing config |
| Control Planes → Edit | decK pushes config changes |

For an org admin account these are already granted.  For a service account,
add these roles in **Settings → Teams** or **Settings → Users**.

### 3.3 What success looks like

After creating the token, you will use it in step 5.  A successful ping
will look like:

```
Successfully connected to Konnect!
Konnect org name:                 my-org
Control Plane name:               entity-risk-ai
```

### 3.4 Troubleshooting

| Symptom | Fix |
|---|---|
| `401 Unauthorized` | Token is wrong or expired — generate a new one |
| `403 Forbidden` | Token lacks the required role — check org/team permissions |
| Konnect UI doesn't show "Personal Access Tokens" | You may be on the legacy plan; use **Organization → API Keys** instead |

---

## 4. Export required environment variables

Add the following to your `.env` file (copy from `.env.example` if you
haven't already), then **restart this notebook kernel** so the new values
are picked up.

```bash
# Kong Konnect targeting
KONG_KONNECT_REGION=au                    # change to match your Konnect org region
KONG_KONNECT_CONTROL_PLANE_NAME=entity-risk-ai
KONG_KONNECT_TOKEN=kpat_XXXXXXXXXXXX      # paste your PAT here

# Enable live tests in this notebook
ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true
```

You do **not** need to set `KONG_PROXY_URL`, `KONG_AI_GATEWAY_ENABLED`, or
any other Kong variable yet.  `KONG_PROXY_URL` will be set in notebook 506
once the Serverless gateway is created and Konnect provides the proxy URL.

In [3]:
# Reload .env and print the current Kong targeting settings (secrets masked).
# Run this after editing .env and restarting the kernel.

load_dotenv("../.env", override=True)
kong = get_kong_settings()
LIVE = kong.notebook_live_tests

m = kong.masked()
print("Kong targeting settings:")
print(f"  konnect_region              = {m['konnect_region'] or '(not set)'}")
print(f"  konnect_control_plane_name  = {m['konnect_control_plane_name'] or '(not set)'}")
print(f"  konnect_token               = {m['konnect_token'] or '(not set)'}")
print(f"  notebook_live_tests         = {LIVE}")

missing = []
if not kong.konnect_region:              missing.append("KONG_KONNECT_REGION")
if not kong.konnect_control_plane_name:  missing.append("KONG_KONNECT_CONTROL_PLANE_NAME")
if not kong.konnect_token:               missing.append("KONG_KONNECT_TOKEN")

if missing:
    print(f"\n[WARN] Missing: {', '.join(missing)}")
    print("       Fill these in .env before running the live connectivity check.")
else:
    print("\n[OK] All targeting variables are set.")

Kong targeting settings:
  konnect_region              = au
  konnect_control_plane_name  = entity-risk-ai
  konnect_token               = kpat_LuC…***
  notebook_live_tests         = True

[OK] All targeting variables are set.


---

## 5. Create a Serverless Gateway control plane in Konnect

A **control plane** is the logical grouping inside Konnect that holds your
routes, services, consumers, and plugins.  For Serverless Gateway, Konnect
also provisions and manages the data plane nodes for you.

### 5.1 Via the Konnect UI

1. In the Konnect left nav, click **Gateway Manager**.
2. Click **+ New Control Plane** (top-right).
3. Choose **Serverless** as the gateway type.
   - _Do not choose "Dedicated Cloud" or "Self-Managed" — those require
     separate data plane nodes._
4. Name it exactly `entity-risk-ai` (matches `KONG_KONNECT_CONTROL_PLANE_NAME`).
5. Choose your region, then click **Create**.
6. Once created, Konnect immediately shows a **proxy URL**
   (e.g. `https://<uuid>.us.gw.konghq.com`).  Copy this — it will
   become `KONG_PROXY_URL` when you reach notebook 506.

If you already have a Serverless control plane you want to reuse, just update
`KONG_KONNECT_CONTROL_PLANE_NAME` to its exact name.

### 5.2 Via decK / CLI

Konnect does not currently support creating control planes via decK directly
(decK syncs config *into* an existing control plane).  Use the UI or the
Konnect Admin API to create it, then use decK for all subsequent config management:

```bash
# List existing control planes
curl -s -H "Authorization: Bearer $KONG_KONNECT_TOKEN" \
     "https://${KONG_KONNECT_REGION}.api.konghq.com/v2/control-planes" \
  | python3 -m json.tool | grep '"name"'
```

### 5.3 What success looks like

After creating the Serverless gateway you should see:
- A green **Active** status badge on the control plane card.
- A **proxy URL** in the Gateway Manager dashboard — copy this for later.
- The control plane appears in the API list above.

There are no "connected data plane nodes" to check — Serverless manages
nodes for you and does not surface them as individual entries in the UI.

In [4]:
# List control planes via the Konnect API.
# LIVE guard: only runs when ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true.

if not LIVE:
    print("[SKIP] Live test disabled. Set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    if not kong.konnect_token or not kong.konnect_region:
        print("[SKIP] KONG_KONNECT_TOKEN or KONG_KONNECT_REGION not set.")
    else:
        api_url = f"https://{kong.konnect_region}.api.konghq.com/v2/control-planes"
        result = subprocess.run(
            ["curl", "-s",
             "-H", f"Authorization: Bearer {kong.konnect_token}",
             api_url],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            print(f"[FAIL] curl error: {result.stderr.strip()}")
        else:
            import json
            try:
                data = json.loads(result.stdout)
                items = data.get("data", [])
                if items:
                    print(f"[OK] Found {len(items)} control plane(s):")
                    for cp in items:
                        cp_type = cp.get("config", {}).get("cluster_type", "unknown")
                        marker = " <-- target" if cp["name"] == kong.konnect_control_plane_name else ""
                        print(f"  - {cp['name']} (type: {cp_type}){marker}")
                else:
                    print("[INFO] No control planes found yet.")
                    print("       Create a Serverless gateway in the Konnect UI first (see section 5.1).")
            except json.JSONDecodeError:
                print("[FAIL] Unexpected response (not JSON):")
                print(result.stdout[:500])

[OK] Found 2 control plane(s):
  - serverless-default (type: CLUSTER_TYPE_SERVERLESS)
  - entity-risk-ai (type: CLUSTER_TYPE_SERVERLESS) <-- target


---

## 6. Deployment model: Konnect Serverless Gateway

This project uses **Konnect Serverless Gateway** throughout phases 505–507.
The table below describes exactly what you do versus what Kong manages:

| You configure | Kong manages |
|---|---|
| Services, routes, plugins (via decK YAML) | Data plane nodes |
| Consumers and API keys | TLS certificates and renewal |
| Rate limits and access control | Horizontal scaling |
| `KONG_PROXY_URL` in your `.env` | The proxy URL itself (provided in Konnect UI) |

### How the proxy URL is obtained

When you create a Serverless gateway in Konnect, the UI immediately shows
a proxy URL like:

```
https://abc123def.au.gw.konghq.com
```

Copy this value.  It goes into `KONG_PROXY_URL` in your `.env` when you
reach notebook 506.  All AI and MCP traffic in phases 506–507 passes through
this URL.

### How config reaches the gateway

```
decK YAML file  →  deck gateway sync  →  Konnect control plane  →  Serverless DP
```

There is no data plane to restart, no certificate to rotate, no container
to rebuild.  Config changes are live within seconds of `deck gateway sync`.

---

## Not needed for this project

The following concepts and steps apply to **hybrid** or **self-managed** Kong
deployments only.  They appear frequently in Kong documentation but are not
relevant here.

| Concept | Why it does not apply here |
|---|---|
| `KONG_CLUSTER_CERT` / `KONG_CLUSTER_CERT_KEY` | Hybrid data plane mTLS — Serverless handles TLS internally |
| `KONG_CLUSTER_CONTROL_PLANE` endpoint | Hybrid clustering — replaced by the Konnect control plane API |
| `docker run kong/kong-gateway` (data plane) | Serverless provisions the DP automatically; no Docker needed |
| `helm install kong` | Kubernetes data plane install — out of scope for this project |
| Manual data plane bootstrap / node registration | Serverless manages all DP nodes internally |
| Dedicated Cloud gateway type in Konnect | Separate paid tier with dedicated infrastructure — not needed here |

If you are reading Kong documentation and encounter these steps, you are
looking at the **Dedicated Cloud** or **Self-Hosted** deployment guides.
You can skip them entirely.

---

## 7. Validate connectivity with decK

### 7.1 Check your decK version

```bash
deck version
```

### 7.2 Ping Konnect

`deck gateway ping` confirms that decK can reach the Konnect control plane
API with your PAT.

```bash
deck gateway ping \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME"
```

Expected output:
```
Successfully connected to Konnect!
Konnect org name:      my-org
Control Plane name:    entity-risk-ai
```

### 7.3 Validate an empty state file

An empty decK state file lets you confirm that decK can parse YAML and read
the control plane state without making any changes:

```bash
cat > /tmp/kong-empty-state.yaml <<'EOF'
_format_version: "3.0"
EOF

deck gateway validate /tmp/kong-empty-state.yaml \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME"
```

Expected output:
```
Validating configuration against Konnect...
Validation successful.
```

In [5]:
# 7.1 — decK version check (local, no LIVE guard needed)

result = subprocess.run(["deck", "version"], capture_output=True, text=True)
if result.returncode == 0:
    print("[PASS] deck version:")
    print(result.stdout.strip())
else:
    print("[FAIL] deck not found — install it first (see section 2).")

[PASS] deck version:
decK v1.55.2 (8bfab74)


In [7]:
# 7.2 — Ping Konnect control plane.
# LIVE guard: only runs when ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true.

if not LIVE:
    print("[SKIP] Live test disabled. Set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
elif not kong.konnect_token:
    print("[SKIP] KONG_KONNECT_TOKEN is not set. Fill it in .env first.")
elif not kong.konnect_addr:
    print("[SKIP] KONG_KONNECT_ADDR is not set. Fill it in .env first.")
elif not kong.konnect_control_plane_name:
    print("[SKIP] KONG_KONNECT_CONTROL_PLANE_NAME is not set. Fill it in .env first.")
else:
    cmd = [
        "deck", "gateway", "ping",
        "--konnect-token", kong.konnect_token,
        "--konnect-addr", kong.konnect_addr,
        "--konnect-control-plane-name", kong.konnect_control_plane_name,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("[PASS] deck gateway ping succeeded:")
        print(result.stdout.strip())
    else:
        print("[FAIL] deck gateway ping failed:")
        print(result.stdout.strip())
        print(result.stderr.strip())
        print("\nTroubleshooting:")
        print("  - Check that KONG_KONNECT_TOKEN is a valid, unexpired PAT.")
        print("  - Check that KONG_KONNECT_ADDR matches your Konnect Address.")
        print("  - Check that KONG_KONNECT_CONTROL_PLANE_NAME matches exactly (case-sensitive).")

[PASS] deck gateway ping succeeded:
Successfully Konnected to the Emil organization!


In [10]:
# 7.3 — Validate an empty decK state file against Konnect.
# Confirms read access without making any changes.
# LIVE guard: only runs when ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true.

import tempfile
import pathlib

if not LIVE:
    print("[SKIP] Live test disabled.")
elif not kong.konnect_token or not kong.konnect_addr or not kong.konnect_control_plane_name:
    print("[SKIP] One or more KONG_KONNECT_* variables are not set.")
else:
    state_yaml = '_format_version: "3.0"\n'
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".yaml", delete=False, prefix="kong-empty-state-"
    ) as f:
        f.write(state_yaml)
        tmp_path = f.name

    cmd = [
        "deck", "gateway", "validate", tmp_path,
        "--konnect-token", kong.konnect_token,
        "--konnect-addr", kong.konnect_addr,
        "--konnect-control-plane-name", kong.konnect_control_plane_name,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    pathlib.Path(tmp_path).unlink(missing_ok=True)

    if result.returncode == 0:
        print("[PASS] deck gateway validate — empty state file is valid:")
        print(result.stdout.strip())
    else:
        print("[FAIL] deck gateway validate failed:")
        print(result.stdout.strip())
        print(result.stderr.strip())

[PASS] deck gateway validate — empty state file is valid:



---

## 8. Expected variables and values

After completing this notebook, your `.env` should have the following
Kong-related lines filled in:

```bash
# Phase 505 — control plane targeting (REQUIRED for 506+)
KONG_KONNECT_REGION=au
KONG_KONNECT_CONTROL_PLANE_NAME=entity-risk-ai
KONG_KONNECT_TOKEN=kpat_XXXXXXXXXXXXXXXX

# Phase 505 — leave blank; set this in notebook 506 once Konnect provides
# the Serverless proxy URL (e.g. https://<uuid>.au.gw.konghq.com)
KONG_PROXY_URL=

# Phase 506+ — leave as false until you start notebook 506
KONG_AI_GATEWAY_ENABLED=false
KONG_MCP_GATEWAY_ENABLED=false

# Notebook gate — set true while running 505+ notebooks, false otherwise
ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true
```

The variables you do **not** need yet:
- `KONG_AI_GATEWAY_ROUTE_PATH` — defaults to `/ai`; set only if you use a different path
- `KONG_AI_GATEWAY_API_KEY` — populated in notebook 506
- `KONG_MCP_GATEWAY_ROUTE_PATH` — defaults to `/mcp`
- `KONG_MCP_GATEWAY_API_KEY` — populated in notebook 507

In [3]:
# Print a summary of all Kong settings showing what is set vs. not set.

load_dotenv("../.env", override=True)
kong = get_kong_settings()
m = kong.masked()

def status(val):
    return "[SET]" if val else "[NOT SET]"

print("Phase 505 — Konnect targeting:")
print(f"  {status(kong.konnect_region):<12} KONG_KONNECT_REGION             = {m['konnect_region'] or '—'}")
print(f"  {status(kong.konnect_addr):<12} KONG_KONNECT_ADDR               = {m['konnect_addr'] or '—'}")
print(f"  {status(kong.konnect_control_plane_name):<12} KONG_KONNECT_CONTROL_PLANE_NAME = {m['konnect_control_plane_name'] or '—'}")
print(f"  {status(kong.konnect_token):<12} KONG_KONNECT_TOKEN              = {m['konnect_token'] or '—'}")
print(f"  {status(kong.konnect_control_plane_id):<12} KONG_KONNECT_CONTROL_PLANE_ID   = {m['konnect_control_plane_id'] or '—'}")
print()
print("Phase 506/507 vars (defined in .env.example, loaded in future notebooks):")
print("  KONG_PROXY_URL, KONG_AI_GATEWAY_*, KONG_MCP_GATEWAY_* — not part of KongSettings yet")
print()
print(f"Notebook gate: ENABLE_LIVE_KONG_NOTEBOOK_TESTS = {kong.notebook_live_tests}")

Phase 505 — Konnect targeting:
  [SET]        KONG_KONNECT_REGION             = au
  [SET]        KONG_KONNECT_ADDR               = https://au.api.konghq.com
  [SET]        KONG_KONNECT_CONTROL_PLANE_NAME = entity-risk-ai
  [SET]        KONG_KONNECT_TOKEN              = kpat_LuC…***
  [SET]        KONG_KONNECT_CONTROL_PLANE_ID   = a92e7da1-469b-4cda-851a-eb326907df7c

Phase 506/507 vars (defined in .env.example, loaded in future notebooks):
  KONG_PROXY_URL, KONG_AI_GATEWAY_*, KONG_MCP_GATEWAY_* — not part of KongSettings yet

Notebook gate: ENABLE_LIVE_KONG_NOTEBOOK_TESTS = True


---

## 9. Troubleshooting

### decK install problems

| Problem | Fix |
|---|---|
| `brew: command not found` | Install Homebrew first: `https://brew.sh` |
| `deck: command not found` after install | Restart your shell or run `export PATH="$(brew --prefix)/bin:$PATH"` |
| Docker decK returns permission error | Add `--user $(id -u)` to the docker run command |
| `unsupported format version` from deck validate | Upgrade decK to ≥ 1.35 (`brew upgrade kong/deck/deck`) |

### Konnect API / PAT problems

| Problem | Fix |
|---|---|
| `401 Unauthorized` from deck ping | Token is wrong or expired — regenerate PAT |
| `403 Forbidden` | PAT lacks control plane Edit permission — check org/team roles |
| `404 Not Found` on control plane name | Name is case-sensitive — check exact spelling in Konnect UI |
| `connection refused` or network timeout | Check corporate proxy / VPN; Konnect requires outbound HTTPS |
| ping succeeds but wrong org listed | Your PAT belongs to a different Konnect org — regenerate with the correct account |

### Wrong region

| Problem | Fix |
|---|---|
| `failed to get org details` | `KONG_KONNECT_REGION` does not match the region your Konnect org was created in |
| Ping succeeds but control plane not found | Control plane is in a different region than specified |

To find your correct region: log into `cloud.konghq.com` → click your org name
in the top-left → the URL changes to `<region>.api.konghq.com` — that is your region.

### Serverless gateway type confusion

| Problem | Fix |
|---|---|
| Konnect UI shows "0 connected data plane nodes" | This is expected for Serverless — nodes are managed internally, not shown individually |
| decK ping works but validate says "Error: listing services" | PAT has View but not Edit — add Control Planes → Edit role |
| You created a Dedicated Cloud or Self-Managed control plane by mistake | Create a new one with type **Serverless**; delete the old one |

### Python / notebook problems

| Problem | Fix |
|---|---|
| `ModuleNotFoundError: src` | Check `sys.path.insert(0, "..")` ran successfully in cell-01 |
| `.env` changes not picked up | Restart the Jupyter kernel and re-run all cells from the top |
| `EnvironmentError` from `get_kong_settings()` | This should never raise — all Kong vars are optional |

---

## 10. Cleanup and rollback

No Kong config was changed by this notebook (only reads and pings).

To clean up completely:

1. **Remove the PAT** — Konnect UI → My Account → Personal Access Tokens → revoke the token.
2. **Delete the control plane** (if you created one just for testing) — Gateway Manager → select CP → Settings → Delete.
3. **Remove Kong vars from `.env`** — delete the `KONG_*` and `ENABLE_LIVE_KONG_NOTEBOOK_TESTS` lines.  The app will continue to run on the direct Anthropic and remote-MCP paths unchanged.

In [4]:
# Minimal rollback helper — removes Kong vars from the in-memory environment
# only (does NOT edit your .env file).
#
# Run this if you want to confirm that the rest of the app still works
# without any Kong variables present.

KONG_VARS = [
    "KONG_KONNECT_REGION",
    "KONG_KONNECT_ADDR",
    "KONG_KONNECT_CONTROL_PLANE_NAME",
    "KONG_KONNECT_TOKEN",
    "KONG_KONNECT_CONTROL_PLANE_ID",
    "ENABLE_LIVE_KONG_NOTEBOOK_TESTS",
]

for var in KONG_VARS:
    os.environ.pop(var, None)

kong_clean = get_kong_settings()
assert not kong_clean.notebook_live_tests
assert not kong_clean.konnect_token
print("[PASS] All Kong vars cleared. get_kong_settings() returns safe defaults.")
print("       The existing Anthropic and remote-MCP paths are completely unaffected.")

[PASS] All Kong vars cleared. get_kong_settings() returns safe defaults.
       The existing Anthropic and remote-MCP paths are completely unaffected.


---

## 11. What comes next — notebooks 506 and 507

Both notebooks build directly on the Serverless Gateway foundation established
here.  **No self-hosted data plane deployment is required for either.**

### Notebook 506 — Kong AI Gateway

- Write a decK YAML file declaring a Kong service and route that fronts the
  Anthropic API on the Serverless gateway.
- Configure the [AI Proxy plugin](https://docs.konghq.com/hub/kong-inc/ai-proxy/)
  so Kong injects the Anthropic API key — the app never holds it.
- Add rate limiting per consumer.
- Set `KONG_PROXY_URL` to the Serverless proxy URL from the Konnect UI.
- Wire `AnthropicClient` to use `KONG_PROXY_URL + KONG_AI_GATEWAY_ROUTE_PATH`
  when `KONG_AI_GATEWAY_ENABLED=true`.
- Smoke-test AI calls through the Serverless gateway.

### Notebook 507 — Kong MCP Gateway

- Expose the FastMCP server behind a Kong route on the same Serverless gateway.
- Configure the Key-Authentication plugin so MCP clients need `X-Kong-API-Key`.
- Map Kong consumers to app roles (`jr_risk_analyst`, `sr_risk_analyst`) using
  ACL scopes that match the tool categories already defined in `policy.py`.
- Replace the mock `authenticate()` in `src/app/auth.py` with a Kong JWT validator.
- Smoke-test MCP calls through the Serverless gateway.

---

### Summary of what this notebook accomplished

| Step | Outcome |
|---|---|
| Deployment model established | Konnect Serverless Gateway — no self-managed DP required |
| Hybrid / self-managed setup steps | Explicitly marked as out of scope |
| `KongSettings` in `src/config.py` | All Kong vars load and mask from a single surface |
| `.env.example` updated | Clear groups for 505, 506, 507 vars |
| decK installed and verified | `deck version` returned success |
| Konnect connectivity validated | `deck gateway ping` confirmed PAT and control plane |
| Safe defaults confirmed | App runs with no Kong vars set — direct paths unaffected |

In [5]:
# Final compile check — confirms no syntax errors were introduced in src/ or app.py.

result = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "../src", "../app.py"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
assert result.returncode == 0, "compileall failed — syntax error in src/ or app.py"
print("[PASS] compileall clean — no syntax errors in src/ or app.py")

[PASS] compileall clean — no syntax errors in src/ or app.py
